# NYC Yellow Taxi Trip Data Analysis

**Dataset:** `yellow_tripdata_2025-11.parquet` (NYC TLC Trip Record Data, November 2025)

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn, NLTK

---
## Loading the Dataset

We start by loading the parquet file and checking its structure.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
print('Setup done')

In [ ]:
df = pd.read_parquet('yellow_tripdata_2025-11.parquet')
print(f'Rows: {df.shape[0]:,}  Columns: {df.shape[1]}')

In [ ]:
df.head(10)

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
# Quick stats on key numeric columns
print(f'Avg fare: ${np.nanmean(df["fare_amount"]):.2f}')
print(f'Median fare: ${np.nanmedian(df["fare_amount"]):.2f}')
print(f'Std fare: ${np.nanstd(df["fare_amount"]):.2f}')
print(f'Avg trip distance: {np.nanmean(df["trip_distance"]):.2f} miles')
print(f'Max tip: ${np.nanmax(df["tip_amount"]):.2f}')
print(f'Total revenue: ${np.nansum(df["total_amount"]):,.2f}')

In [ ]:
# Check value counts for categorical fields
print('VendorID distribution:')
print(df['VendorID'].value_counts())
print('\nPayment type distribution:')
print(df['payment_type'].value_counts())

The dataset has millions of records with 19 columns covering fares, distances, timestamps and location IDs. Looking at the value counts, one vendor handles the bulk of trips and credit card is the dominant payment type.

---
## Data Cleaning and Visualization

Check for missing values, remove invalid records, then visualize the main distributions.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Plotting libs ready')

In [ ]:
# Check missing values
nulls = df.isnull().sum()
nulls_pct = (df.isnull().sum() / len(df) * 100).round(2)
null_df = pd.DataFrame({'Missing': nulls, 'Pct': nulls_pct})
null_df[null_df['Missing'] > 0]

In [ ]:
# Clean the data
df_clean = df.copy()

# Drop rows where passenger_count is null or 0
df_clean = df_clean[df_clean['passenger_count'].notna()]
df_clean = df_clean[df_clean['passenger_count'] > 0]

# Remove negative/zero fares and unreasonable values
df_clean = df_clean[df_clean['fare_amount'] > 0]
df_clean = df_clean[df_clean['fare_amount'] < 500]
df_clean = df_clean[df_clean['trip_distance'] > 0]
df_clean = df_clean[df_clean['trip_distance'] < 100]
df_clean = df_clean[df_clean['total_amount'] > 0]

# Remove negative tips
df_clean = df_clean[df_clean['tip_amount'] >= 0]

print(f'Before cleaning: {len(df):,} rows')
print(f'After cleaning: {len(df_clean):,} rows')
print(f'Removed: {len(df) - len(df_clean):,} rows ({(len(df)-len(df_clean))/len(df)*100:.1f}%)')

In [ ]:
# Fare distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df_clean['fare_amount'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Fare Amount Distribution')
axes[0].set_xlabel('Fare ($)')
axes[0].set_ylabel('Count')

axes[1].hist(df_clean['fare_amount'][df_clean['fare_amount'] < 60], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Fare Amount (Zoomed < $60)')
axes[1].set_xlabel('Fare ($)')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numeric columns
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

sns.boxplot(y=df_clean['trip_distance'][df_clean['trip_distance'] < 30], ax=axes[0], color='lightblue')
axes[0].set_title('Trip Distance')

sns.boxplot(y=df_clean['fare_amount'][df_clean['fare_amount'] < 80], ax=axes[1], color='lightgreen')
axes[1].set_title('Fare Amount')

sns.boxplot(y=df_clean['tip_amount'][df_clean['tip_amount'] < 20], ax=axes[2], color='lightyellow')
axes[2].set_title('Tip Amount')

plt.tight_layout()
plt.show()

In [ ]:
# Payment type bar chart
pay_labels = {0:'Flex Fare', 1:'Credit Card', 2:'Cash', 3:'No Charge', 4:'Dispute', 5:'Unknown', 6:'Voided'}
pay_counts = df_clean['payment_type'].map(pay_labels).value_counts()

plt.figure(figsize=(10, 5))
pay_counts.plot(kind='bar', color=sns.color_palette('Set2', len(pay_counts)), edgecolor='black')
plt.title('Payment Type Distribution')
plt.xlabel('Payment Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
numeric_cols = ['fare_amount','trip_distance','tip_amount','tolls_amount',
    'total_amount','congestion_surcharge','passenger_count','extra','mta_tax']

plt.figure(figsize=(10, 8))
corr = df_clean[numeric_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Trips by hour of day
df_clean['pickup_hour'] = df_clean['tpep_pickup_datetime'].dt.hour

hourly = df_clean['pickup_hour'].value_counts().sort_index()
plt.figure(figsize=(12, 5))
plt.bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
plt.title('Number of Trips by Hour of Day')
plt.xlabel('Hour')
plt.ylabel('Trip Count')
plt.xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# Trips by day of week
df_clean['pickup_day'] = df_clean['tpep_pickup_datetime'].dt.day_name()
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
daily = df_clean['pickup_day'].value_counts().reindex(day_order)

plt.figure(figsize=(10, 5))
plt.bar(daily.index, daily.values, color='coral', edgecolor='white')
plt.title('Trips by Day of Week')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

After cleaning we removed a small fraction of bad records (negative fares, zero passengers etc). The fare histogram is clearly right-skewed, most trips cost under $20. Credit card dominates as payment method. The heatmap shows trip_distance and fare_amount are highly correlated, and tip_amount also tracks fare well. Peak taxi usage is during evening rush hours, and weekdays see more trips than weekends.

---
## Descriptive Statistics

Compute summary statistics, look at skewness/kurtosis, and compare across groups.

In [ ]:
df_clean.describe().round(2)

In [ ]:
# Skewness and Kurtosis
from scipy import stats

num_cols = ['fare_amount','trip_distance','tip_amount','total_amount','tolls_amount','passenger_count']
sk = df_clean[num_cols].skew().round(3)
kt = df_clean[num_cols].kurtosis().round(3)

pd.DataFrame({'Skewness': sk, 'Kurtosis': kt})

In [ ]:
# Mean, Median, Mode side by side
stats_df = pd.DataFrame({
    'Mean': df_clean[num_cols].mean().round(2),
    'Median': df_clean[num_cols].median().round(2),
    'Mode': df_clean[num_cols].mode().iloc[0].round(2),
    'Std': df_clean[num_cols].std().round(2),
    'Variance': df_clean[num_cols].var().round(2)
})
stats_df

In [ ]:
# Group by VendorID
vendor_stats = df_clean.groupby('VendorID')[['fare_amount','trip_distance','tip_amount','total_amount']].agg(['mean','median','std']).round(2)
vendor_stats

In [ ]:
# Distribution plots with mean/median markers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, col in zip(axes.flat, ['fare_amount','trip_distance','tip_amount','total_amount']):
    data = df_clean[col][df_clean[col] < df_clean[col].quantile(0.95)]
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.7)
    ax.axvline(data.mean(), color='red', linestyle='--', label=f'Mean: {data.mean():.2f}')
    ax.axvline(data.median(), color='green', linestyle='-', label=f'Median: {data.median():.2f}')
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Percentile analysis for fare_amount
percentiles = [5, 10, 25, 50, 75, 90, 95, 99]
fare_pcts = np.percentile(df_clean['fare_amount'], percentiles)
pct_df = pd.DataFrame({'Percentile': percentiles, 'Fare ($)': fare_pcts.round(2)})
print('Fare Amount Percentiles:')
pct_df

All fare-related columns show positive skewness and high kurtosis which is common for financial data. The mean is above the median in every case, confirming the right skew (pulled up by expensive trips). Both vendors show similar per-trip averages. The percentile table gives a clear picture of fare spread.

---
## Feature Scaling, Encoding and Selection

Normalize the numeric features with different scalers and encode categorical variables for ML.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.feature_selection import VarianceThreshold

# Prepare numeric feature matrix
features = ['trip_distance','fare_amount','tip_amount','tolls_amount','extra',
    'mta_tax','congestion_surcharge','total_amount','passenger_count']

# Take a sample for efficiency
df_sample = df_clean[features + ['VendorID','RatecodeID','payment_type']].dropna().sample(50000, random_state=42)
print(f'Sample size: {len(df_sample):,}')

In [ ]:
# StandardScaler
scaler_std = StandardScaler()
X_std = pd.DataFrame(scaler_std.fit_transform(df_sample[features]), columns=features)
print('StandardScaler - Mean and Std after scaling:')
print(X_std.describe().loc[['mean','std']].round(4))

In [ ]:
# MinMaxScaler
scaler_mm = MinMaxScaler()
X_mm = pd.DataFrame(scaler_mm.fit_transform(df_sample[features]), columns=features)
print('MinMaxScaler - Min and Max after scaling:')
print(X_mm.describe().loc[['min','max']].round(4))

# Compare distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(df_sample['fare_amount'], bins=40, color='gray', edgecolor='white')
axes[0].set_title('Original fare_amount')
axes[1].hist(X_std['fare_amount'], bins=40, color='steelblue', edgecolor='white')
axes[1].set_title('StandardScaler')
axes[2].hist(X_mm['fare_amount'], bins=40, color='coral', edgecolor='white')
axes[2].set_title('MinMaxScaler')
plt.tight_layout()
plt.show()

In [ ]:
# Label Encoding for categorical variables
le_vendor = LabelEncoder()
le_rate = LabelEncoder()
le_pay = LabelEncoder()

df_sample['VendorID_enc'] = le_vendor.fit_transform(df_sample['VendorID'])
df_sample['RatecodeID_enc'] = le_rate.fit_transform(df_sample['RatecodeID'].fillna(99).astype(int))
df_sample['payment_type_enc'] = le_pay.fit_transform(df_sample['payment_type'].fillna(0).astype(int))

# One-hot encoding for payment_type
pay_dummies = pd.get_dummies(df_sample['payment_type'].astype(int), prefix='pay')
print('One-hot encoded payment columns:')
pay_dummies.head()

In [ ]:
# Correlation-based feature selection
corr_with_total = df_sample[features].corrwith(df_sample['total_amount']).abs().sort_values(ascending=False)
print('Feature correlation with total_amount:')
print(corr_with_total.round(3))

plt.figure(figsize=(10, 5))
corr_with_total.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Feature Correlation with Total Amount')
plt.ylabel('|Correlation|')
plt.axhline(y=0.1, color='red', linestyle='--', label='Threshold 0.1')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Variance Threshold
selector = VarianceThreshold(threshold=0.1)
X_var = selector.fit_transform(X_std)
selected = [features[i] for i in range(len(features)) if selector.get_support()[i]]
removed = [features[i] for i in range(len(features)) if not selector.get_support()[i]]
print(f'Features kept ({len(selected)}): {selected}')
print(f'Features removed ({len(removed)}): {removed}')

StandardScaler centers data to mean=0, std=1 while MinMaxScaler squeezes everything into [0,1]. The bar chart shows which features correlate most with total_amount. fare_amount naturally has the highest correlation since it is a component of total. Variance threshold keeps most features since the taxi data has enough spread.

---
## Classification: Predicting High Tips

Create a binary target (is the tip above the median?) and train several classifiers.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import precision_score, recall_score, f1_score

In [ ]:
# Prepare classification data: predict high tip (above median)
clf_features = ['trip_distance','fare_amount','passenger_count','extra',
    'mta_tax','tolls_amount','congestion_surcharge']

df_clf = df_clean[clf_features + ['tip_amount']].dropna().sample(50000, random_state=42)
median_tip = df_clf['tip_amount'].median()
df_clf['high_tip'] = (df_clf['tip_amount'] > median_tip).astype(int)

X = df_clf[clf_features]
y = df_clf['high_tip']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Target balance: {y.value_counts().to_dict()}')
print(f'Median tip threshold: ${median_tip:.2f}')

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
print('Logistic Regression:')
print(classification_report(y_test, y_pred_lr))

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=10, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
print('Decision Tree:')
print(classification_report(y_test, y_pred_dt))

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
print('Random Forest:')
print(classification_report(y_test, y_pred_rf))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, preds, title in zip(axes, [y_pred_lr, y_pred_dt, y_pred_rf],
    ['Logistic Regression', 'Decision Tree', 'Random Forest']):
    ConfusionMatrixDisplay.from_predictions(y_test, preds, ax=ax, cmap='Blues')
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# Comparison table
models = {'Logistic Regression': y_pred_lr, 'Decision Tree': y_pred_dt, 'Random Forest': y_pred_rf}
results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test, preds), 4),
        'Precision': round(precision_score(y_test, preds, average='weighted', zero_division=0), 4),
        'Recall': round(recall_score(y_test, preds, average='weighted'), 4),
        'F1': round(f1_score(y_test, preds, average='weighted'), 4)
    })
pd.DataFrame(results)

In [ ]:
# Feature importance from Random Forest
importance = pd.Series(rf.feature_importances_, index=clf_features).sort_values(ascending=True)

plt.figure(figsize=(10, 5))
importance.plot(kind='barh', color='teal')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

The comparison table above shows Random Forest gives the best numbers across all metrics. fare_amount and trip_distance are the top features for predicting tips, which makes sense since longer and pricier trips tend to get larger tips.

---
## Regression: Predicting Total Amount

Train regression models to predict total_amount from trip features and compare their performance.

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Prepare regression data
reg_features = ['trip_distance','fare_amount','tip_amount','passenger_count',
    'extra','mta_tax','tolls_amount','congestion_surcharge']

df_reg = df_clean[reg_features + ['total_amount']].dropna().sample(50000, random_state=42)
X_r = df_reg[reg_features]
y_r = df_reg['total_amount']

X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X_r, y_r, test_size=0.2, random_state=42)
print(f'Train: {len(X_train_r)}, Test: {len(X_test_r)}')

In [ ]:
# Linear Regression
lr_reg = LinearRegression()
lr_reg.fit(X_train_r, y_train_r)
y_pred_lr_r = lr_reg.predict(X_test_r)
print('Linear Regression:')
print(f'  R2: {r2_score(y_test_r, y_pred_lr_r):.4f}')
print(f'  RMSE: {np.sqrt(mean_squared_error(y_test_r, y_pred_lr_r)):.4f}')
print(f'  MAE: {mean_absolute_error(y_test_r, y_pred_lr_r):.4f}')

In [ ]:
# Ridge Regression
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_r, y_train_r)
y_pred_ridge = ridge.predict(X_test_r)

# Lasso Regression
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_r, y_train_r)
y_pred_lasso = lasso.predict(X_test_r)

print('Ridge R2:', round(r2_score(y_test_r, y_pred_ridge), 4))
print('Lasso R2:', round(r2_score(y_test_r, y_pred_lasso), 4))

In [ ]:
# Decision Tree Regressor
dtr = DecisionTreeRegressor(max_depth=15, random_state=42)
dtr.fit(X_train_r, y_train_r)
y_pred_dtr = dtr.predict(X_test_r)

# Random Forest Regressor
rfr = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rfr.fit(X_train_r, y_train_r)
y_pred_rfr = rfr.predict(X_test_r)

print('Decision Tree R2:', round(r2_score(y_test_r, y_pred_dtr), 4))
print('Random Forest R2:', round(r2_score(y_test_r, y_pred_rfr), 4))

In [ ]:
# Comparison table
reg_results = []
for name, preds in [('Linear', y_pred_lr_r), ('Ridge', y_pred_ridge),
    ('Lasso', y_pred_lasso), ('Decision Tree', y_pred_dtr), ('Random Forest', y_pred_rfr)]:
    reg_results.append({
        'Model': name,
        'R2': round(r2_score(y_test_r, preds), 4),
        'RMSE': round(np.sqrt(mean_squared_error(y_test_r, preds)), 4),
        'MAE': round(mean_absolute_error(y_test_r, preds), 4)
    })
pd.DataFrame(reg_results)

In [ ]:
# Actual vs Predicted scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test_r, y_pred_lr_r, alpha=0.3, s=5, color='steelblue')
axes[0].plot([0, 100], [0, 100], 'r--')
axes[0].set_title('Linear Regression: Actual vs Predicted')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')
axes[0].set_xlim(0, 100)
axes[0].set_ylim(0, 100)

axes[1].scatter(y_test_r, y_pred_rfr, alpha=0.3, s=5, color='teal')
axes[1].plot([0, 100], [0, 100], 'r--')
axes[1].set_title('Random Forest: Actual vs Predicted')
axes[1].set_xlabel('Actual')
axes[1].set_ylabel('Predicted')
axes[1].set_xlim(0, 100)
axes[1].set_ylim(0, 100)

plt.tight_layout()
plt.show()

In [ ]:
# Residual analysis for Linear Regression
residuals = y_test_r - y_pred_lr_r

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_pred_lr_r, residuals, alpha=0.3, s=5, color='coral')
axes[0].axhline(y=0, color='black', linestyle='-')
axes[0].set_title('Residuals vs Predicted')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Residuals')

axes[1].hist(residuals, bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')

plt.tight_layout()
plt.show()

All models get very high R2 here because total_amount is basically the sum of the other fare components (fare + tip + tolls + surcharges), so the relationship is close to linear. Linear, Ridge and Lasso give nearly identical results. The residual plot is centered around 0 which is good.

---
## Cross-Validation and Hyperparameter Tuning

Evaluate models with k-fold cross-validation and search for better hyperparameters.

In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV, learning_curve

# Cross-validation on classification models
cv_results = {}
for name, model in [('Logistic Regression', LogisticRegression(max_iter=1000, random_state=42)),
                    ('Decision Tree', DecisionTreeClassifier(max_depth=10, random_state=42)),
                    ('Random Forest', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1))]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    cv_results[name] = scores
    print(f'{name}: mean={scores.mean():.4f} std={scores.std():.4f}')


In [ ]:
# GridSearchCV on Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=3, scoring='accuracy', verbose=0, n_jobs=-1)
grid_search.fit(X_train, y_train)

print('Best params:', grid_search.best_params_)
print('Best CV accuracy:', round(grid_search.best_score_, 4))
print('Test accuracy:', round(grid_search.score(X_test, y_test), 4))

In [ ]:
# RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 200),
    'max_depth': randint(5, 25),
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5)
}

random_search = RandomizedSearchCV(RandomForestClassifier(random_state=42, n_jobs=-1),
    param_dist, n_iter=15, cv=3, scoring='accuracy', random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

print('Best params:', random_search.best_params_)
print('Best CV accuracy:', round(random_search.best_score_, 4))
print('Test accuracy:', round(random_search.score(X_test, y_test), 4))

In [ ]:
# Learning curve for best model
train_sizes, train_scores, val_scores = learning_curve(
    grid_search.best_estimator_, X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 8), cv=3, scoring='accuracy', n_jobs=-1)

plt.figure(figsize=(10, 5))
plt.plot(train_sizes, train_scores.mean(axis=1), 'o-', label='Training score')
plt.plot(train_sizes, val_scores.mean(axis=1), 's-', label='Validation score')
plt.fill_between(train_sizes, train_scores.mean(axis=1)-train_scores.std(axis=1),
    train_scores.mean(axis=1)+train_scores.std(axis=1), alpha=0.1)
plt.fill_between(train_sizes, val_scores.mean(axis=1)-val_scores.std(axis=1),
    val_scores.mean(axis=1)+val_scores.std(axis=1), alpha=0.1)
plt.title('Learning Curve (Tuned Random Forest)')
plt.xlabel('Training Set Size')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# CV score comparison plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(cv_results.values(), labels=cv_results.keys())
ax.set_title('Cross-Validation Accuracy Comparison')
ax.set_ylabel('Accuracy')
plt.tight_layout()
plt.show()

Cross-validation confirms the Random Forest is the most reliable classifier with the smallest variance across folds. GridSearchCV and RandomizedSearchCV both land on similar optimal parameters. The learning curve shows training and validation scores converging, so the model is not overfitting badly.

---
## Grouping Trips with K-Means Clustering

Use K-Means to segment trips into groups based on distance, fare, tip and passenger count.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Prepare clustering features
clust_features = ['trip_distance','fare_amount','tip_amount','passenger_count','tolls_amount']
df_clust = df_clean[clust_features].dropna().sample(20000, random_state=42)

scaler_c = StandardScaler()
X_clust = scaler_c.fit_transform(df_clust)
print(f'Clustering on {len(df_clust)} samples, {len(clust_features)} features')

In [ ]:
# Elbow method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)

plt.figure(figsize=(10, 5))
plt.plot(K_range, inertias, 'bo-')
plt.title('Elbow Method')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Silhouette scores
sil_scores = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust)
    score = silhouette_score(X_clust, labels, sample_size=5000)
    sil_scores.append(score)
    print(f'k={k}: silhouette={score:.4f}')

plt.figure(figsize=(10, 5))
plt.plot(range(2, 9), sil_scores, 'go-')
plt.title('Silhouette Score vs K')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# KMeans with k=4 (from the elbow/silhouette above)
optimal_k = 4
km_final = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
clusters = km_final.fit_predict(X_clust)
df_clust['cluster'] = clusters

print('Cluster sizes:')
print(df_clust['cluster'].value_counts().sort_index())

In [ ]:
# PCA for 2D visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_clust)

plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='Set2', alpha=0.5, s=10)
plt.colorbar(scatter, label='Cluster')
plt.title(f'K-Means Clusters (k={optimal_k}) - PCA Projection')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.tight_layout()
plt.show()

In [ ]:
# Cluster profiles
print('Cluster means (original scale):')
df_clust.groupby('cluster')[clust_features].mean().round(2)

The elbow plot bends around k=4 and the silhouette scores support that. Looking at the cluster means, the groups roughly separate out short cheap trips, medium-distance rides, longer expensive rides, and trips with tolls (likely airport). Silhouette scores are moderate which is normal for messy real-world data.

---
## Density-Based Clustering with DBSCAN

DBSCAN finds clusters based on point density and can identify noise/outlier points automatically.

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors

# Use a smaller sample (DBSCAN is expensive)
X_db = X_clust[:5000]  # already scaled from above

In [ ]:
# K-distance graph to find a good eps
nn = NearestNeighbors(n_neighbors=5)
nn.fit(X_db)
distances, _ = nn.kneighbors(X_db)
distances = np.sort(distances[:, -1])

plt.figure(figsize=(10, 5))
plt.plot(distances)
plt.title('K-Distance Graph (k=5)')
plt.xlabel('Points (sorted)')
plt.ylabel('5th Nearest Neighbor Distance')
plt.axhline(y=1.5, color='r', linestyle='--', label='eps=1.5')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# DBSCAN clustering
dbscan = DBSCAN(eps=1.5, min_samples=10)
db_labels = dbscan.fit_predict(X_db)

n_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_noise = list(db_labels).count(-1)

print(f'DBSCAN found {n_clusters} clusters')
print(f'Noise points: {n_noise} ({n_noise/len(db_labels)*100:.1f}%)')
print(f'Cluster sizes: {pd.Series(db_labels).value_counts().to_dict()}')

In [ ]:
# Visualize DBSCAN vs KMeans
pca_db = PCA(n_components=2)
X_pca_db = pca_db.fit_transform(X_db)

km_compare = KMeans(n_clusters=4, random_state=42, n_init=10)
km_labels = km_compare.fit_predict(X_db)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].scatter(X_pca_db[:, 0], X_pca_db[:, 1], c=km_labels, cmap='Set2', alpha=0.5, s=10)
axes[0].set_title('K-Means (k=4)')

colors = db_labels.copy().astype(float)
colors[db_labels == -1] = np.nan
axes[1].scatter(X_pca_db[:, 0], X_pca_db[:, 1], c=colors, cmap='Set2', alpha=0.5, s=10)
noise_mask = db_labels == -1
axes[1].scatter(X_pca_db[noise_mask, 0], X_pca_db[noise_mask, 1],
    c='black', marker='x', s=15, alpha=0.5, label='Noise')
axes[1].set_title('DBSCAN')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Try different eps values
for eps in [0.8, 1.0, 1.5, 2.0, 2.5]:
    db = DBSCAN(eps=eps, min_samples=10)
    labels = db.fit_predict(X_db)
    n_c = len(set(labels)) - (1 if -1 in labels else 0)
    n_n = list(labels).count(-1)
    print(f'eps={eps}: {n_c} clusters, {n_n} noise points ({n_n/len(labels)*100:.1f}%)')

DBSCAN picks up the dense core of normal trips and marks unusual ones (very long, very expensive, etc.) as noise. Smaller eps means stricter density requirements and more noise points. Compared to KMeans, DBSCAN does not force every point into a cluster, so it is better at spotting anomalous trips.

---
## Generative AI Overview

Generative AI covers models that can produce new content like text, images or code. Large Language Models (LLMs) such as GPT, Gemini and LLaMA are built on the Transformer architecture.

### Key Concepts

- **Transformers:** Self-attention mechanism that processes sequences in parallel, enabling training on huge text corpora
- **LLMs:** Neural networks pre-trained on massive text data, then fine-tuned for specific tasks
- **Prompt Engineering:** Writing input prompts carefully to get better outputs from LLMs
- **Few-shot Learning:** Including examples in the prompt to guide the model's response
- **Temperature:** Controls randomness in generation, lower = more deterministic, higher = more creative

In [ ]:
# Simple n-gram based text generation to show the core idea
# - predict next token based on previous ones
import random
from collections import defaultdict

# Generate trip descriptions as training corpus
trip_texts = []
for _, row in df_clean.sample(1000, random_state=42).iterrows():
    dist = row['trip_distance']
    fare = row['fare_amount']
    pax = int(row.get('passenger_count', 1)) if pd.notna(row.get('passenger_count')) else 1
    if dist < 2: trip_type = 'short'
    elif dist < 5: trip_type = 'medium'
    else: trip_type = 'long'
    trip_texts.append(f'a {trip_type} trip with {pax} passengers fare {int(fare)} dollars')

# Build bigram model
bigrams = defaultdict(list)
for text in trip_texts:
    words = text.split()
    for i in range(len(words)-1):
        bigrams[words[i]].append(words[i+1])

# Generate text
def generate(start_word, length=10):
    result = [start_word]
    word = start_word
    for _ in range(length):
        if word in bigrams:
            word = random.choice(bigrams[word])
            result.append(word)
        else:
            break
    return ' '.join(result)

random.seed(42)
print('Generated trip descriptions (bigram model):')
for _ in range(5):
    print(f'  > {generate("a", 8)}')

### Prompt Engineering Examples

When using an LLM for data analysis, prompt structure matters:

**Zero-shot:** *"Analyze this taxi trip data and identify patterns."*

**Few-shot:** *"Given trip data where distance=2.5mi, fare=$12, tip=$3, this is a standard Manhattan trip. Now analyze: distance=18mi, fare=$52, tip=$0"*

**Chain-of-thought:** *"Step 1: Check the fare vs distance ratio. Step 2: Check if tip is proportional. Step 3: Identify the likely route type..."*

These techniques help LLMs produce more structured and accurate outputs for data tasks.

The bigram model above generates somewhat coherent trip descriptions by learning which words follow which. Real LLMs use transformer attention with billions of parameters for much better text generation. Prompt engineering techniques like few-shot and chain-of-thought can improve the quality of LLM outputs.